In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high_q5/checkpoints/post_cell_38_try_1.pickle

In [ ]:
%%RecordEvent
%%time
### cell 39 ###

# restore the question string so it’s in scope
question_of_interest_cell51 = (
    'Which of the following natural language processing (NLP) methods do you use on a regular basis?'
)


def grab_subset_of_data_51(df, question):
    # pick out columns that match the question
    mask = df.columns.str.contains(question)
    cols = df.columns[mask]
    # split on the separator ' - ' (not on every '-') to get the full option text
    suffixes = cols.str.split(' - ', 1).str[1].str.strip()
    # subset & rename to just the option suffix
    sub = df[cols].rename(columns=dict(zip(cols, suffixes)))
    # drop rows where all of these option columns are null
    return sub.dropna(how='all', subset=suffixes.tolist())


def compute_transformer_pct(df, question):
    sub = grab_subset_of_data_51(df, question)
    # count non-null per option, sort ascending, then compute percentage
    counts = sub.count().sort_values(ascending=True)
    pct = counts.mul(100).div(len(sub))
    # rename any option whose text starts with 'Transformer' to a uniform 'Transformers'
    transformer_keys = [opt for opt in pct.index if opt.lower().startswith('transformer')]
    pct = pct.rename({k: 'Transformers' for k in transformer_keys})
    # return just the Transformer percentage
    return pct[['Transformers']]

# compute once per year
percentages_2019_cell51 = compute_transformer_pct(responses_df_2019_cell10, question_of_interest_cell51)
percentages_2020_cell51 = compute_transformer_pct(responses_df_2020,             question_of_interest_cell51)
percentages_2021_cell51 = compute_transformer_pct(responses_df_2021,             question_of_interest_cell51)
percentages_2022_cell51 = compute_transformer_pct(responses_df_2022_cell10,      question_of_interest_cell51)

# display info as before
(percentages_2019_cell51.info(),
 percentages_2020_cell51.info(),
 percentages_2021_cell51.info(),
 percentages_2022_cell51.info())

In [ ]:
%Checkpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high_q5/checkpoints/post_cell_39_try_3.pickle

In [ ]:
%PrintCellInfo opt_cell_exec_info

In [ ]:

with open("/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/opt_cell_exec_info_39_try_3.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[39], f)


In [ ]:
opt_output = Out.get(4)